# A8 — v20 별표 주입 파일럿

기존 company_size 한 호출의 시스템 프롬프트 끝에 제공 지침 **제2조와 별표 1 원문만** 붙입니다.
스키마·소비자·관측 게이트는 그대로입니다. 각 군 dev 200건, 회차당 400응답입니다.
기본/SME는 보관 응답인 **혼합 CPU 재생**이며 전체 GPU 또는 서버 점수가 아닙니다.

1. A100 GPU와 HF_TOKEN 보안 비밀을 준비합니다.
2. 실행 안내의 `a8-v20-inputs.zip`을 Drive `MyDrive/a8/`에 업로드합니다.
3. 회차 1은 `EPISODE=1`. 위에서부터 실행하고 마지막 셀에서 ZIP을 받습니다.
4. 회차 1 완료·시간 통과 후 런타임을 삭제하고 새 A100에서 같은 노트북·ATTEMPT로 `EPISODE=2`.
5. 실패 재시도는 ATTEMPT를 바꾸고 회차 1부터 시작합니다. 실패해도 마지막 ZIP 셀을 실행합니다.

실행기는 **생성 호출 전에** 이 런타임의 실제 토크나이저로 두 군의 전체 메시지를 세고,
추가 축소가 1건이라도 있으면 `budget.json`을 남기고 멈춥니다.
company 단계 상한은 군별 339.178초/200건입니다. 새 후보의 실제 속도·성능은 미측정입니다.

In [ ]:
import csv
import gzip
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

WORK = Path(tempfile.mkdtemp(prefix="a8-v20-", dir="/content"))
RESULTS = WORK / "results"
RESULTS.mkdir()
PYTHON = str(WORK / "venv/bin/python")
MODEL_ID = "google/gemma-4-26B-A4B-it"
REVISION = "4d7ae4984b7db7de8f8457170b3f1a419ee76d52"
EXPECTED_PACKAGES = {"vllm": "0.26.0", "torch": "2.11.0+cu130",
                     "transformers": "5.14.1", "xgrammar": "0.2.3"}
SERVER_PYTHON = "3.12.13"
case_inputs = {}
REPO_URL = "https://github.com/LittleBitAI/ai-nara-shop.git"
REPO_REF = "d2c5f54a9365a76de9d938970dd34018ac823058"  # 실행기·후보·검사를 포함한 고정 코드

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n",
                    encoding="utf-8", newline="\n")

def run_logged(name, command, env=None, cwd=None):
    started = time.time()
    entry = {"argv": [str(x) for x in command], "cwd": str(cwd or WORK), "started": started,
             "returncode": None}
    if entry["argv"][0] == PYTHON:
        # Absolute Python paths do not activate PATH for tools such as ninja.
        env = dict(os.environ if env is None else env)
        env["PATH"] = str(Path(PYTHON).parent) + os.pathsep + env.get("PATH", "")
        env["VIRTUAL_ENV"] = str(Path(PYTHON).parent.parent)
    log_path = RESULTS / (name + ".log")
    record_path = RESULTS / (name + "-command.json")
    if log_path.exists() or record_path.exists():
        raise ValueError(f"{name} 실행 기록이 이미 있습니다. 첫 셀부터 새 작업 폴더로 실행하세요.")
    write_json(record_path, entry)
    try:
        with log_path.open("x", encoding="utf-8", newline="\n") as log_file:
            with subprocess.Popen(entry["argv"], cwd=entry["cwd"], env=env, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                  errors="replace", bufsize=1) as process:
                try:
                    for line in process.stdout:
                        log_file.write(line)
                        log_file.flush()
                        print(line, end="")
                    entry["returncode"] = process.wait()
                except BaseException:
                    process.terminate()
                    try:
                        process.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        process.kill()
                    raise
    finally:
        entry["elapsed_seconds"] = time.time() - started
        write_json(record_path, entry)
    if entry["returncode"] != 0:
        raise RuntimeError(f"{name} 실패 (exit={entry['returncode']}). 마지막 로그 다운로드 셀을 실행하세요.")
    return log_path

write_json(RESULTS / "host.json", {"python": sys.version, "work": str(WORK)})
print("작업 폴더:", WORK)

## 코드 고정

In [ ]:
if len(REPO_REF) != 40 or any(c not in "0123456789abcdef" for c in REPO_REF):
    raise ValueError("실행 안내(run-request.md)의 40자리 커밋 SHA로 REPO_REF를 고정하세요.")
REPO = WORK / "repo"
run_logged("git-clone", ["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
run_logged("git-fetch", ["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO)
run_logged("git-checkout", ["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if SOURCE_COMMIT != REPO_REF:
    raise ValueError("REPO_REF를 실행 안내의 40자리 SHA로 고정하세요.")
SUBMISSION = REPO  # 기존 고정 환경 설치 셀에서 requirements.txt 경로로만 사용
write_json(RESULTS / "source.json", {"commit": SOURCE_COMMIT, "requested_ref": REPO_REF})
assert (REPO / "experiments/a5_scope_pilot.py").is_file()
assert (REPO / "experiments/a8_v20_annex.py").is_file()
assert (REPO / "experiments/law_index.py").is_file()

## Drive 입력과 회차 선택

clone만으로 입력이 생긴다고 가정하지 않습니다. 준비된 입력 ZIP에서 명세에 있는 파일만
복원하고 모든 SHA256을 검사합니다. 무라벨 20,000건 파일은 필요하지 않습니다.
제공 법령 원문은 주입 대상이므로 같은 명세로 함께 검사합니다.
회차 2는 회차 1의 완료·시간을 먼저 검사하며, 실행기는 코드·입력·설정·환경도 대조합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
EPISODE = 1  # 새 런타임 반복은 2
ATTEMPT = "a"  # 실패 후 새 시도는 b 등. 두 회차는 같은 값.
assert EPISODE in (1, 2)
assert ATTEMPT.isalnum() and len(ATTEMPT) <= 16
INPUT_BUNDLE = Path("/content/drive/MyDrive/a8/a8-v20-inputs.zip")
PILOT_ROOT = Path("/content/drive/MyDrive/a8/v20-annex-" + SOURCE_COMMIT[:12] + "-" + ATTEMPT)
FACTS_OUTPUT = PILOT_ROOT / ("episode-" + str(EPISODE))
if FACTS_OUTPUT.exists():
    raise FileExistsError("회차 결과가 이미 있습니다. EPISODE 또는 ATTEMPT를 확인하세요.")
if EPISODE == 2:
    previous = json.loads((PILOT_ROOT / "episode-1/summary.json").read_text(encoding="utf-8"))
    report = json.loads((PILOT_ROOT / "episode-1/run_report.json").read_text(encoding="utf-8"))
    assert previous["complete"] and report["status"] == "complete", "회차 1 미완료"
    assert all(a["within_stage_planning_limit"] for a in previous["arms"].values()), "회차 1 시간 상한 초과"
expected = json.loads((REPO / "reports/team-c/a8-v20-annex/inputs.json").read_text(encoding="utf-8"))
missing = [name for name in expected if not (REPO / name).is_file()]
if missing:
    if not INPUT_BUNDLE.is_file():
        write_json(RESULTS / "input-check.json", {"status": "missing", "files": missing})
        raise FileNotFoundError("a8-v20-inputs.zip을 Drive의 내 드라이브/a8에 업로드하세요.")
    with zipfile.ZipFile(INPUT_BUNDLE) as archive:
        for name in missing:
            target = (REPO / name).resolve()
            if not target.is_relative_to(REPO.resolve()):
                raise ValueError("입력 경로가 저장소 밖입니다.")
            data = archive.read(name)
            if hashlib.sha256(data).hexdigest() != expected[name]:
                raise ValueError("입력 ZIP SHA256 불일치: " + name)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
for name, digest in expected.items():
    with (REPO / name).open("rb") as source:
        if hashlib.file_digest(source, "sha256").hexdigest() != digest:
            raise ValueError("입력 SHA256 불일치: " + name)
write_json(RESULTS / "input-check.json", {"status": "verified", "files": expected, "episode": EPISODE})
print("고정 코드:", SOURCE_COMMIT, "이번 결과:", FACTS_OUTPUT)

## 2. 실제 GPU와 디스크 확인

GPU 종류·드라이버·VRAM·RAM을 기록합니다. 30GiB VRAM/80GiB 여유 디스크는 이 노트북의 사전 거름 기준이며 적재 보장이 아닙니다. 부족하면 모델 다운로드 전에 멈춥니다.

In [ ]:
gpu_log = run_logged("gpu", ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                             "--format=csv,noheader"])
gpus = gpu_log.read_text(encoding="utf-8").strip().splitlines()
if not gpus or max(int(row.split(",")[1].strip().split()[0]) for row in gpus) < 30 * 1024:
    raise RuntimeError("VRAM 30GiB 미만입니다. 더 큰 GPU 런타임을 선택한 뒤 처음부터 실행하세요.")
resources = {"gpus": gpus, "disk_free_bytes": shutil.disk_usage(WORK).free,
             "meminfo": Path("/proc/meminfo").read_text()}
write_json(RESULTS / "resources.json", resources)
if resources["disk_free_bytes"] < 80 * 1024**3:
    raise RuntimeError("설치와 원본 모델을 준비할 여유 디스크 80GiB가 필요합니다.")
print("GPU 사전 확인 통과. 실제 적재/실행 성공은 아직 확인하지 않았습니다.")

## 고정 Python·추론 패키지 설치

기존 Colab 검증과 같은 Python 3.12.13·vLLM 0.26.0·CUDA 13.0을 사용합니다.

In [ ]:
# Colab 커널 Python과 분리해 대회 서버의 정확한 Python 버전을 준비합니다.
run_logged("uv-install", [sys.executable, "-m", "pip", "install", "uv"])
run_logged("python-install", [sys.executable, "-m", "uv", "venv", "--python", SERVER_PYTHON,
                             "--seed", str(WORK / "venv")])
run_logged("install", [PYTHON, "-m", "pip", "install",
    "--extra-index-url", "https://download.pytorch.org/whl/cu130",
    *[f"{name}=={version}" for name, version in EXPECTED_PACKAGES.items()]])
# 같은 저장소의 requirements.txt를 설치합니다.
run_logged("submission-install", [PYTHON, "-m", "pip", "install", "-r",
                                  str(SUBMISSION / "requirements.txt")])
if json.loads((RESULTS / "submission-install-command.json").read_text())["elapsed_seconds"] > 600:
    raise RuntimeError("제출 requirements 설치가 서버 제한 600초를 넘었습니다.")
run_logged("pip-freeze", [PYTHON, "-m", "pip", "freeze"])
runtime_code = """
import json, platform, shutil, subprocess, sys, torch, vllm, transformers, xgrammar
from importlib.metadata import version
from pathlib import Path
runtime = {"python": platform.python_version(), "cuda": torch.version.cuda,
           "packages": {n: version(n) for n in ["torch", "vllm", "transformers", "xgrammar"]},
           "gpu": torch.cuda.get_device_name(0),
           "ninja_path": shutil.which("ninja"),
           "ninja_version": subprocess.check_output(["ninja", "--version"], text=True).strip()}
Path(sys.argv[1]).write_text(json.dumps(runtime, indent=2) + "\\n", encoding="utf-8")
print(runtime)
torch.empty(1, device="cuda")
"""
run_logged("runtime", [PYTHON, "-c", runtime_code, str(RESULTS / "runtime.json")])
runtime = json.loads((RESULTS / "runtime.json").read_text())
if (runtime["python"] != SERVER_PYTHON or runtime["packages"] != EXPECTED_PACKAGES
        or runtime["cuda"] != "13.0"):
    raise RuntimeError("서버의 Python/핵심 패키지/CUDA 빌드와 다릅니다. 검증을 중단합니다.")

## 4. HF_TOKEN으로 고정 리비전 모델 다운로드

Colab 보안 비밀에 `HF_TOKEN`을 등록하고 이 노트북의 접근을 허용하세요. 읽지 못하면 숨김 입력으로 받으며 **빈 토큰은 거부**합니다.
토큰은 모델 다운로드 자식 프로세스 환경변수로만 전달하고, 추론 환경·명령행·로그·결과 ZIP에 기록하지 않습니다.

[고정 모델](https://huggingface.co/google/gemma-4-26B-A4B-it/tree/4d7ae4984b7db7de8f8457170b3f1a419ee76d52)의 접근 권한이 있는 계정을 사용합니다.
가중치는 준비 단계에서만 내려받고 제출물에 포함하지 않습니다.

In [ ]:
from google.colab import userdata
from getpass import getpass

try:
    token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    token = getpass("HF_TOKEN (Hugging Face 읽기 토큰, 필수): ")
if not token or not token.strip():
    raise ValueError("HF_TOKEN이 필요합니다. Colab 보안 비밀에 등록하고 노트북 접근을 허용하세요.")
download_env = {**os.environ, "HF_TOKEN": token.strip()}
download_env.pop("HF_HUB_OFFLINE", None)
download_env.pop("TRANSFORMERS_OFFLINE", None)
download_code = """
from huggingface_hub import snapshot_download
from pathlib import Path
import os
import sys
model_dir = snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], token=os.environ["HF_TOKEN"],
    allow_patterns=["*.safetensors", "*.json", "*.jinja", "*.model"])
if Path(model_dir).name != sys.argv[2]:
    raise RuntimeError("고정 snapshot 경로 불일치")
Path(sys.argv[3]).write_text(model_dir, encoding="utf-8")
"""
try:
    run_logged("model-download", [PYTHON, "-c", download_code, MODEL_ID, REVISION,
                                 str(WORK / "model-path.txt")], env=download_env)
finally:
    download_env.pop("HF_TOKEN", None)
    token = None
MODEL_DIR = (WORK / "model-path.txt").read_text(encoding="utf-8")
write_json(RESULTS / "model.json", {"id": MODEL_ID, "revision": REVISION, "path": MODEL_DIR})

## 대조군·후보 실행과 혼합 재생

회차 1 control→a8, 새 런타임 회차 2 a8→control. 후보는 프롬프트만 다르므로
같은 저장 응답을 후보 문맥 OFF/ON으로 재생하면 24항목 CSV가 바이트까지 같아야 합니다.
control↔후보와 같은 군 두 회차도 비교합니다. 군별 `off-hybrid.csv`/`on-hybrid.csv`,
24항목 지표·오답·원응답·검증 이유가 저장됩니다.
시간 초과도 실패로 남기며 중간 결과를 보존합니다. 실패 시 마지막 ZIP 셀을 실행하세요.

In [ ]:
inference_env = dict(os.environ)
inference_env.pop("HF_TOKEN", None)
inference_env.pop("HUGGING_FACE_HUB_TOKEN", None)
run_logged("a8-v20-annex", [PYTHON, str(REPO / "experiments/a5_scope_pilot.py"),
    "--experiment", "a8", "--output-dir", str(FACTS_OUTPUT),
    "--model-dir", MODEL_DIR, "--episode", str(EPISODE)], env=inference_env, cwd=REPO)
summary = json.loads((FACTS_OUTPUT / "summary.json").read_text(encoding="utf-8"))
budget = json.loads((FACTS_OUTPUT / "budget.json").read_text(encoding="utf-8"))
print(json.dumps({key: budget[key] for key in
                  ("token_count_kind", "reference_chars", "added_tokens_min", "added_tokens_max",
                   "additional_shrink", "visible_differs", "control_truncated",
                   "conditions_pass", "budget_safety_evidence")}, ensure_ascii=False, indent=2))
print(json.dumps({name: {"hybrid_macro_f1": arm["metrics"]["on"]["macro_f1"],
                       "v20": arm["metrics"]["on"]["items"]["v20"],
                       "stage_seconds": arm["dev_stage_seconds"],
                       "within_limit": arm["within_stage_planning_limit"]}
                  for name, arm in summary["arms"].items()}, ensure_ascii=False, indent=2))
print("회차 1 완료. ZIP을 보관하고 새 런타임에서 회차 2를 실행하세요." if EPISODE == 1 else
      "두 회차 결과 ZIP을 전달하세요. 채택·전체 GPU·서버 검증은 미완료입니다.")

## 결과 ZIP 다운로드 — 실패했어도 이 셀 실행

설치/실행 로그와 같은 ATTEMPT의 회차 1·2 원응답·혼합 재생·비교를 묶습니다.
중간 events 로그와 partial 파일도 보존합니다. 모델 가중치나 HF_TOKEN은 포함하지 않습니다.

In [ ]:
from google.colab import files

archive_path = WORK / ("a8-v20-results-" + str(time.time_ns()) + ".zip")
with zipfile.ZipFile(archive_path, "x", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file():
            archive.write(path, "logs/" + path.relative_to(RESULTS).as_posix())
    if "PILOT_ROOT" in globals() and PILOT_ROOT.exists():
        for path in sorted(PILOT_ROOT.rglob("*")):
            if path.is_file() and path.suffix != ".zip":
                archive.write(path, "pilot/" + path.relative_to(PILOT_ROOT).as_posix())
if "PILOT_ROOT" in globals():
    PILOT_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive_path, PILOT_ROOT / archive_path.name)
print("결과 ZIP:", archive_path)
files.download(str(archive_path))